In [ ]:
# ============================================
# 1️⃣ Mount Google Drive
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import os
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import copy
import random
import warnings

warnings.filterwarnings("ignore", message=".*can only test a child process.*")
warnings.filterwarnings("ignore", message=".*Bad file descriptor.*")

# ============================================
# 2️⃣ Login & Base Settings
# ============================================
wandb.login()
PROJECT_NAME = "alexnet_100x_experiments_2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# ============================================
# 3️⃣ Dataset from Google Drive
# ============================================
# ❗❗ SET THIS TO YOUR ZIP OR FOLDER PATH
DATASET_PATH = "/content/drive/My Drive/Data1/antsbeesdataset/"

local_path = "/content/data"
os.makedirs(local_path, exist_ok=True)

# If ZIP file → unzip it
if DATASET_PATH.endswith(".zip"):
    print("📦 Unzipping dataset from Google Drive...")
    !unzip -q "{DATASET_PATH}" -d "{local_path}"
else:
    print("📁 Copying dataset folder...")
    !cp -r "{DATASET_PATH}" "{local_path}"

print("📂 Dataset ready at:", local_path)

# ============================================
# 3.1️⃣ Auto-detect train/valid folders
# ============================================
def find_dir(base, keywords):
    for root, dirs, _ in os.walk(base):
        for d in dirs:
            low = d.lower()
            if any(k in low for k in keywords):
                return os.path.join(root, d)
    return None

train_dir = find_dir(local_path, ["train"])
valid_dir = find_dir(local_path, ["val", "valid"])

print("🔍 Found folders:")
print(" train_dir:", train_dir)
print(" valid_dir:", valid_dir)

if train_dir is None or valid_dir is None:
    raise ValueError("❌ 'train' or 'valid' directory not found. Check ZIP structure!")

# ============================================
# 3.2️⃣ Dataloaders
# ============================================
input_size = 224
batch_size = 32

train_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

valid_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
valid_dataset = datasets.ImageFolder(valid_dir, transform=valid_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"✅ Train Samples: {len(train_dataset)}  |  Valid Samples: {len(valid_dataset)}")
print(f"📚 Classes: {train_dataset.classes}")

# ============================================
# 4️⃣ Training Function
# ============================================
def run_experiment(run_id, experiment_type, weights=None, epochs=10, lr=1e-4):
    run = wandb.init(
        project=PROJECT_NAME,
        name=f"{experiment_type}_run_{run_id:03d}",
        config={
            "architecture": "AlexNet",
            "experiment_type": experiment_type,
            "run_id": run_id,
            "epochs": epochs,
            "learning_rate": lr,
            "batch_size": batch_size,
        },
        reinit=True,
    )

    # Model setup
    if weights == "imagenet":
        model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
    else:
        model = models.alexnet(weights=None)

    model.classifier[6] = nn.Linear(model.classifier[6].in_features, len(train_dataset.classes))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    # Epoch loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_correct = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            total_loss += loss.item() * inputs.size(0)
            total_correct += torch.sum(preds == labels.data)

        train_loss = total_loss / len(train_loader.dataset)
        train_acc = total_correct.double() / len(train_loader.dataset)

        # Validation
        model.eval()
        val_loss, val_correct = 0.0, 0

        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                val_loss += loss.item() * inputs.size(0)
                val_correct += torch.sum(preds == labels.data)

        valid_loss = val_loss / len(valid_loader.dataset)
        valid_acc = val_correct.double() / len(valid_loader.dataset)

        wandb.log({
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "train/accuracy": train_acc,
            "valid/loss": valid_loss,
            "valid/accuracy": valid_acc,
        })

        if valid_acc > best_acc:
            best_acc = valid_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), f"{experiment_type}_best_{run_id}.pth")

    model.load_state_dict(best_model_wts)
    wandb.log({"best_validation_accuracy": best_acc})
    wandb.finish()

    print(f"🏁 Run {run_id} ({experiment_type}) | Best Acc: {best_acc:.4f}")

# ============================================
# 5️⃣ 100-Run Experiment Loop
# ============================================
EXPERIMENT_TYPES = ["pretrained", "untrained", "baseline"]

for run_id in range(1, 101):
    exp_type = random.choice(EXPERIMENT_TYPES)
    weights = "imagenet" if exp_type != "untrained" else None
    print(f"\n🚀 Starting run {run_id}/100 — {exp_type}")
    run_experiment(run_id, exp_type, weights=weights, epochs=30, lr=1e-4)

print("\n🎉 All 100 runs complete!")
